In [ ]:
# Import various libraries 

import numpy as np
import astropy
import photutils
import ccdproc
from ccdproc import CCDData, combiner
from astropy import units as u
import astropy.io.fits as fits
from astropy.io import ascii

# This is used to debayer data - we won't be using it today
# import cv2

import astroalign as aa

from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
import matplotlib.pyplot as plt
from photutils.centroids import centroid_com, centroid_1dg, centroid_2dg
from photutils.aperture import CircularAperture
from photutils.aperture import aperture_photometry
from photutils.detection import DAOStarFinder
from photutils.background import Background2D
from photutils.segmentation  import detect_sources, deblend_sources, SourceCatalog
from scipy.ndimage import shift
import gc                               

from astropy.coordinates import SkyCoord
from astroquery.gaia import Gaia


## Displaying images

We are going to be displaying a bunch of images this week so let's have a function to do it.

In [ ]:
# What is this function doing?

def quickimdisplay(image, vmin, vmax):
    # What is this code doing? 
    if vmin==vmax:
        vmin=np.percentile(image, 5)
        vmax=np.percentile(image, 95)    
    # Standard imshow command
    plt.imshow(image, origin='lower', vmin=vmin, vmax=vmax, cmap='grey')
    plt.xlabel('x pixels')
    plt.ylabel('y pixels')
    # Size the colour bar
    s = len(image)/len(image[0])
    if s>1:
        s=1
    plt.colorbar(shrink=s, label='Counts')
    plt.show()

# Combining images 

In the Week 2 lab we combined images after shifting, subtracting and scaling but didn't really have the opportunity to explore what happens when things go wrong.

Today we will create some artifical data to test combine operations. This data with have 2D Guassian stars and Guassian random background noise.

In [ ]:

# define normalized 2D gaussian
# This code is adapted from https://stackoverflow.com/questions/7687679/how-to-generate-2d-gaussian-with-python

def gaus2d(x=0, y=0, mx=0, my=0, sx=4, sy=4):

    g = np.exp(-((x - mx)**2. / (2. * sx**2.) + (y - my)**2. / (2. * sy**2.)))
    g = g / (2.0*np.pi*sx * sy)

    return g

x = np.linspace(0, 100, 100)
y = np.linspace(0, 100, 100)
x, y = np.meshgrid(x, y) # get 2D variables instead of 1D
z = gaus2d(x, y, 50.0, 50.0)
print('Sum over Guassian: ', np.sum(z))
plt.title('Gaussian')
quickimdisplay(z, 0.0, np.max(z))


noise = np.random.normal(1000, 30, (100, 100))
plt.title('Random noise')
quickimdisplay(noise, 0.0, 0.0)



# Lets create artifical images


In [ ]:
idx=0       # Image index
imsize=100  # Image size
im=[]       # A blank list of images 

# Loop through the following code till the index is 9
while idx<9:

    # Create an image with random background noise
    # The mean background value is dropping by 50 counts per image
    im.append(np.random.normal(1000-50*idx, 30, (imsize, imsize)))

    x = np.linspace(0, 100, imsize)
    y = np.linspace(0, 100, imsize)
    x, y = np.meshgrid(x, y) # get 2D variables instead of 1D

    # Add a star with a total of 3000 counts - (1.0 + 0.03*idx) is mock attenuation for rising object
    z = 3000 * gaus2d(x, y, 20, 50) * (1.0 + 0.03*idx)
    print(len(z))
    im[idx]=im[idx]+z

    # Add a star with a total of 6000 counts - (1.0 + 0.03*idx) is mock attenuation for rising object
    z = 6000 * gaus2d(x, y, 50, 50) * (1.0 + 0.03*idx)
    im[idx]=im[idx]+z

    # Add a star with a total of 12000 counts - (1.0 + 0.03*idx) is mock attenuation for rising object
    z = 12000 * gaus2d(x, y, 80, 50) * (1.0 + 0.03*idx)
    im[idx]=im[idx]+z

    # Add a nasty satellite trail
    im[idx][10+8*idx:10+8*idx+1]=np.median(im[idx])+100
    
    plt.rcParams["figure.figsize"] = (3,3)
    plt.title('Image: ' + str(idx))
    quickimdisplay(im[idx], 0.0, 0.0)
    
    idx=idx+1

## Let's combine the images and do everything right 

In [ ]:
# Create a list of images that are background subtracted and scaled
tim=[]   # Temp image list
for idx, image in enumerate(im):
    t=image.copy()          # Make a copy of the image
    t=t-np.ma.median(t)     # Background subtract 
    t=t / (1.0 + 0.03*idx)  # Perfect scaling using the first image as the reference
    tim.append(t)           # Append the list 

image_mean_data = np.mean(np.dstack(tim), -1)

image_median_data = np.median(np.dstack(tim), -1)

plt.rcParams["figure.figsize"] = (3,3)
plt.title('Mean image')
quickimdisplay(image_mean_data, 0.0, 0.0)

plt.rcParams["figure.figsize"] = (3,3)
plt.title('Median image')
quickimdisplay(image_median_data, 0.0, 0.0)

positions = [(20.0, 50.0), (50.0, 50.0), (80.0, 50.0)]
aperture = CircularAperture(positions, r=20.0)

print('\n\n')
print('Mean background: ', np.median(image_mean_data))
print('Mean standard dev: ', np.std(image_mean_data))
print('Mean photometry (has built in background subtraction)')
phot_table_mean = aperture_photometry(image_mean_data-np.median(image_mean_data), aperture)
print(phot_table_mean)

print('\n\n')
print('Median background: ', np.median(image_median_data))
print('Median standard dev: ', np.std(image_median_data))
print('Median photometry (has built in background subtraction)')
phot_table_median = aperture_photometry(image_median_data-np.median(image_median_data), aperture)
print(phot_table_median)



### Question: How does the photometry compare to the inputs?

### <font color='blue'>Answer </font>

Now copy the code from above but lets now skip the background subtraction. What then happens?

### Question: What happens when we skip background subtraction? Why does this happen?

### <font color='blue'>Answer </font>


Now copy the code from above but lets now skip the scaling. What then happens?

### Question: What happens when we skip scaling? Why does this happen?

### <font color='blue'>Answer </font>



### Question: How could choice of star or skipping background subtraction impact scaling images before combining?

### <font color='blue'>Answer </font>


# SeeStar

For the projects this year you will be using data from SeeStar smart telescopes. These small telescopes are equipped with CMOS sensors with a Bayer matrix, which enables CMOS (and CCD) detectors to take colour images. 

## Bayer matrix 

An image of a Bayer matrix lifted directly from wikipedia.

The diagnal grid we see in the individual exposures is a result of the Bayer matrix, effectively a grid of filters on our detector.

<IMG SRC='Bayer_pattern_on_sensor.png'> 

## Let's display some example data straight from a SeeStar telescope

Stacked_15_M 41_10.0s_IRCUT_20250228-223556.fit is a combined image produced by the SeeStar

Light_M 41_10.0s_IRCUT_20250228-222923.fit is an individual exposure in the M41 sub folder



In [ ]:
# Opening the stacked image 

filename='Stacked_15_M 41_10.0s_IRCUT_20250228-223556.fit'
stackim = fits.open(filename)[0]  # What is the zero doing here?
print(stackim)
stackim.header

# Whole image 
quickimdisplay(stackim.data[0], 0.0, 0.0)

# Zoom in 
quickimdisplay(stackim.data[0][0:50, 0:50], 0.0, 0.0)


### Question: How many dimensions does the stacked image have?

### <font color='blue'>Answer </font>

### Question: What are each of the dimensions?

### <font color='blue'>Answer </font>


In [ ]:
# Opening the individual exposure image 

filename='M_41_sub/Light_M 41_10.0s_IRCUT_20250228-222923.fit'
im = fits.open(filename)[0] 
print(im)
im.header


### Question: How many dimensions does the individual exposure image have?

### <font color='blue'>Answer </font>

### Question: How is the colour information stored in the individual images? (To answer this it is probably easiest to look at the data)

### <font color='blue'>Answer </font>



### Question: The SeeStar combined images come with a world coordinate system (WCS). If the default SeeStar combined image and our combined image use the same reference image (i.e. the first image), what can we do to obtain a world coordinate system for our own combined image?

### <font color='blue'>Answer </font>


In [ ]:
# Here's the code we use to debayer images.
# We won't use it in this workshop as you (probably) don't have cv2 so it won't run
# Plus I've haven't set up the inputs (e.g. im as a seires of individual exposures)

i=0 # Switch so this code doesn't run and generate errors 
if i==1:
    im_r=[]
    im_g=[]
    im_b=[]
    for im in scim:
        tim_r = im.copy()   # Create copies of the orginal image  
        tim_g = im.copy()
        tim_b = im.copy()
        temp = cv2.cvtColor(im.data, cv2.COLOR_BayerGRBG2RGB) # DeBayer the original image 
        tim_r.data = temp[:,:,0]  # Red channel to red image 
        tim_g.data = temp[:,:,1]  # Green channel to green image 
        tim_b.data = temp[:,:,2]  # Blue channel to blue image 
    
        im_r.append(tim_r) # Append image lists 
        im_g.append(tim_g)
        im_b.append(tim_b)
    

In [ ]:
## Let's take a look at some data that has been debayered

In [ ]:
filename='M_41_sub/Light_M 41_10.0s_IRCUT_20250228-222934.fit'
im = CCDData.read(filename, unit="adu")

dirname='M_41_sub_debayer/'

bimages = ccdproc.ImageFileCollection(dirname,glob_include='*_b.fit')
im_b = [CCDData.read(dirname+filename, unit="adu") for filename in bimages.files_filtered()]

gimages = ccdproc.ImageFileCollection(dirname,glob_include='*_g.fit')
im_g = [CCDData.read(dirname+filename, unit="adu") for filename in gimages.files_filtered()]

rimages = ccdproc.ImageFileCollection(dirname,glob_include='*_r.fit')
im_r = [CCDData.read(dirname+filename, unit="adu") for filename in rimages.files_filtered()]


In [ ]:
# Comment - what is this code for
xmin=600
xmax=xmin+50
ymin=600
ymax=ymin+50

# Comment
plt.title('Single exposure subregion')
quickimdisplay(im.data[ymin:ymax,xmin:xmax], 0.0, 0.0)

plt.title('Single blue exposure subregion')
quickimdisplay(im_b[0].data[ymin:ymax,xmin:xmax], 0.0, 0.0)

plt.title('Single green exposure subregion')
quickimdisplay(im_g[0].data[ymin:ymax,xmin:xmax], 0.0, 0.0)

plt.title('Single red exposure subregion')
quickimdisplay(im_r[0].data[ymin:ymax,xmin:xmax], 0.0, 0.0)



### Question: How do the images compare?

### <font color='blue'>Answer </font>

### Question: Lets now display the first and last green images? How have they moved? 

It may make sense to zoom in the center of the images.

### <font color='blue'>Answer </font>


## Calibrating with Gaia Photometry 

I can plot SeeStar fluxes as a function of Gaia photometry to determine the relationship between SeeStar magnitude and Gaia photometry. You need a lot of stars to do this well, such as what you find in an open cluster.

<IMG SRC='seestar_gaia_cal.png' width=400>

For calibrated SeeStar magnitudes the red calibration is given by:

$r_{SeeStar} = zp - 2.5\times {\rm log}(f)$

The y axis in the plot is given by...

$y = G_{RP} + 2.5\times {\rm log}(f)$

...so...

$y = G_{RP} + zp - r_{SeeStar} $

$zp = y - (G_{RP} - r_{SeeStar})$

Which doesn't seem that helpful. That said, it does tell us that the plot is showing the difference between Gaia and SeeStar magnitudes as a function of Gaia colour. 

## Question: Is there a situation where we already know the calibrated SeeStar magnitude for a star or stars? 

### <font color='blue'>Answer </font>


## Calibration

So just as we could determine relationships between BVR photometry and Gaia photometry in last weeks lab, using the approach discussed above and other obervations we get the following relations:

These equations are for $G_{BP}-G<1$ or $G-G_{RP}<1$ respectively.

SeeStar b-band:

\begin{equation}
b_{SeeStar} = G_{BP} + 0.385 \times (G_{BP}-G)
\end{equation}

SeeStar g-band:

\begin{equation}
g_{SeeStar} = G + 0.155 \times (G_{BP}-G)^2 + 0.443 \times (G_{BP}-G)
\end{equation}

SeeStar r-band:

\begin{equation}
r_{SeeStar} = G_{RP} + 0.370 \times (G-G_{RP})^2 + 0.572 \times (G-G_{RP})
\end{equation}


# Backgrounds 

Sometimes we don't have a uniform background and need to subtract a gradient. This is often not a flat field error as the variation in the background does not reflect a varying sensitivity to light from celestial sources. Instead it reflects scattered light or varying light pollution across the field of view, both of which tend to impact the SeeStar telescopes (although it can be an issue for all telescopes).

To model the varying background we can use photutils Background2D.

In [ ]:
# Plot a single g-band exposures
plt.title('g exposure')
quickimdisplay(im_g[0].data, 0.0, 0.0)

# Model the data using a 100x100 box size
im_bkg = Background2D(im_g[0], 100)
# Data
print(im_bkg.data)
# Background
print(im_bkg.background)

# Plot the background - Note the scale
plt.title('g background')
quickimdisplay(im_bkg.background.value, 0.0, 0.0)


## Question: the background image can sometimes slightly follow the celestial sources? This is not ideal but is it a significant problem in this case (and how can one check)?

### <font color='blue'>Answer </font>


## Question: play with varying the box size for the background subtraction. How may a very large or very small box size change the background subtraction?

### <font color='blue'>Answer </font>
